# Orbit: fine-tune Gemma 3 270M to sort journal entries into bubbles

This notebook is self-contained: it writes every source file it needs. Google Drive only has to hold the data:

```
MyDrive/orbit/dataset/orbit_bubbles_v3.train.jsonl
MyDrive/orbit/dataset/orbit_bubbles_v2.clean.val.jsonl
```

**Run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.

Every output goes to `MyDrive/orbit/runs/<RUN_NAME>/` as it is produced: a checkpoint after each epoch, `eval.json`, the merged model and the `.litertlm` export. If Colab disconnects, reconnect and **Run all** again: finished steps are skipped and training resumes from the last epoch.

For a new experiment, change the settings in step 0 (at minimum `RUN_NAME`) and Run all.

## 0. Settings

In [ ]:
RUN_NAME = 'orbit-lora-v3'     # new name = new run folder; same name = resume / reuse
FULL_FINETUNE = False          # True trains all 270M weights instead of a LoRA (then use LR = 5e-5)
EPOCHS = 5
RANK = 32                      # LoRA rank (ignored for full fine-tune)
LR = 2e-4
BATCH, GRAD_ACCUM = 4, 4       # 16 examples per step; fits a T4
TRAIN_FILE = 'orbit_bubbles_v3.train.jsonl'
VAL_FILE = 'orbit_bubbles_v2.clean.val.jsonl'   # keep fixed so runs are comparable
BASE_MODEL = 'unsloth/gemma-3-270m-it'          # ungated copy of google/gemma-3-270m-it

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, json
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/orbit/dataset'
RUN = f'/content/drive/MyDrive/orbit/runs/{RUN_NAME}'
os.makedirs(RUN, exist_ok=True)
for name in (TRAIN_FILE, VAL_FILE):
    path = f'{DATA}/{name}'
    assert os.path.exists(path), f'Upload {name} to MyDrive/orbit/dataset/ first'
    print(f'{name}: {sum(1 for _ in open(path))} examples')
os.makedirs('/content/orbit', exist_ok=True)
%cd /content/orbit
!pip install -q -U "transformers==5.19.0" peft accelerate

## 2. Source files
Generated from the repo by `build_notebook.py`; edit the repo files and rebuild rather than editing here.

In [ ]:
%%writefile orbit_schema.py
"""Shared definitions for the Orbit bubble-sorting dataset.

One example = a raw journal entry in, a JSON list of category bubbles out.
Every bubble text must be copied verbatim from the entry.
"""
import json
import os
import re
from pathlib import Path
from typing import Any

# Must match the instruction the app sends at inference time.
INSTRUCTION = "Sort this journal entry into category bubbles. Use only the writer's exact words. Return JSON."

CATEGORIES = {
    "Work": "job, boss, coworkers, shifts, career, job hunting",
    "School": "classes, exams, homework, grades, lecturers, school fees",
    "Money": "rent, bills, spending, saving, debt, salary, prices",
    "Family": "parents, siblings, relatives, own children, parenting, caring for family",
    "Relationships": "romantic partner, dating, crushes, breakups, marriage",
    "Friends": "friends, social life, fitting in, feeling left out by friends",
    "Health & Body": "sickness, pain, sleep, exercise, food and eating, body image, energy",
    "Feelings": "present emotions stated as feelings: sad, happy, angry, tired, numb, burned out",
    "Worries": "what-ifs, anxiety, fear about things that might happen",
    "Self-Reflection": "self-talk, self-esteem, identity, weighing decisions, values, noticing own patterns",
    "Goals & Plans": "intentions, resolutions, habits to build, things to learn, future plans",
    "Gratitude": "thankfulness, appreciation, counting blessings",
    "Hobbies & Creativity": "music, art, writing, sports for fun, games, shows, crafts",
    "Faith & Spirituality": "church, mosque, prayer, God, meditation, spiritual life",
    "Daily Life": "chores, errands, groceries, home, commute, ordinary logistics",
    "Grief & Loss": "death, mourning, missing someone who died, losing something important",
    "Other": "anything that clearly fits none of the other categories",
}

_WORD = re.compile(r"\w")


def load_env(path: Path = Path(__file__).parent / ".env") -> None:
    """Load KEY=value lines from .env into os.environ (existing variables win)."""
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            key, sep, value = line.partition("=")
            if sep and not line.lstrip().startswith("#"):
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
_SENTENCE_END = re.compile(r"[.!?…]\W*$")


def user_message(entry: str) -> str:
    return f"{INSTRUCTION}\n\n{entry}"


def _locate(entry: str, spans: list[str]) -> list[tuple[int, int]] | None:
    """Assign each span a non-overlapping position in the entry, earliest first.

    Returns (start, end) per span in input order, or None if no such assignment exists.
    """
    order = sorted(range(len(spans)), key=lambda i: entry.find(spans[i]))
    taken: list[tuple[int, int]] = []
    result: list[tuple[int, int] | None] = [None] * len(spans)
    for i in order:
        span, pos = spans[i], entry.find(spans[i])
        while pos != -1 and any(pos < e and pos + len(span) > s for s, e in taken):
            pos = entry.find(span, pos + 1)
        if pos == -1:
            return None
        taken.append((pos, pos + len(span)))
        result[i] = (pos, pos + len(span))
    return result  # type: ignore[return-value]


def check_example(entry: str, bubbles: object, require_full_coverage: bool = True) -> tuple[list[dict] | None, str | None]:
    """Validate one example. Returns (canonical_bubbles, None) or (None, reason).

    Canonical form: texts in entry order within each bubble, bubbles ordered by
    their first text's position, and same-category neighbours within a sentence
    merged, so the model sees one consistent target for each entry.
    """
    if not isinstance(entry, str) or not entry.strip():
        return None, "empty entry"
    if not isinstance(bubbles, list) or not bubbles:
        return None, "no bubbles"

    seen_categories: set[str] = set()
    flat: list[tuple[str, str]] = []  # (category, span)
    for bubble in bubbles:
        if not isinstance(bubble, dict) or set(bubble) != {"category", "texts"}:
            return None, f"malformed bubble: {bubble!r}"
        category, texts = bubble["category"], bubble["texts"]
        if category not in CATEGORIES:
            return None, f"unknown category: {category!r}"
        if category in seen_categories:
            return None, f"duplicate category bubble: {category}"
        seen_categories.add(category)
        if not isinstance(texts, list) or not texts:
            return None, f"empty bubble: {category}"
        for text in texts:
            if not isinstance(text, str) or not text.strip():
                return None, f"empty text in {category}"
            if text != text.strip():
                return None, f"leading/trailing whitespace: {text!r}"
            if text not in entry:
                return None, f"not an exact substring: {text!r}"
            flat.append((category, text))

    positions = _locate(entry, [span for _, span in flat])
    if positions is None:
        return None, "spans overlap or reuse the same words"

    if require_full_coverage:
        covered = [False] * len(entry)
        for start, end in positions:
            covered[start:end] = [True] * (end - start)
        gap = "".join(ch for ch, c in zip(entry, covered) if not c)
        if _WORD.search(gap):
            return None, f"words left out of every bubble: {gap.strip()!r}"

    # Merge neighbouring same-category spans inside one sentence, so a sentence is
    # never chopped into pieces that all land in the same bubble.
    pieces = sorted((start, end, category) for (category, _), (start, end) in zip(flat, positions))
    merged: list[list] = []
    for start, end, category in pieces:
        if merged and merged[-1][2] == category:
            prev_end = merged[-1][1]
            between = entry[prev_end:start]
            if not re.search(r"[\w.!?…]", between) and not _SENTENCE_END.search(entry[merged[-1][0]:prev_end]):
                merged[-1][1] = end
                continue
        merged.append([start, end, category])

    by_category: dict[str, list[str]] = {}
    for start, end, category in merged:
        by_category.setdefault(category, []).append(entry[start:end])
    canonical = [{"category": c, "texts": texts} for c, texts in by_category.items()]  # first-appearance order
    return canonical, None


_SENTENCE = re.compile(r".+?(?:[.!?…]+(?=\s|$)|$)", re.S)


def _sentences(entry: str) -> list[tuple[int, int]]:
    """(start, end) of each sentence, whitespace trimmed."""
    spans = []
    for match in _SENTENCE.finditer(entry):
        start, end = match.span()
        while start < end and entry[start].isspace():
            start += 1
        while end > start and entry[end - 1].isspace():
            end -= 1
        if _WORD.search(entry[start:end]):
            spans.append((start, end))
    return spans


def repair_bubbles(entry: str, bubbles: object) -> list[dict] | None:
    """Turn a near-miss model reply into bubbles that pass check_example.

    The model's choices are kept wherever they are usable; the fixes are mechanical:
    - a span with stray whitespace or a small rewrite snaps to the closest sentence,
    - a span that repeats words already claimed loses the repeated part (first wins),
    - duplicate category bubbles merge, unknown categories become Other,
    - words left out join the bubble of the text right before them (or after, at the start).
    Returns None only when nothing in the reply can be placed in the entry.
    """
    from difflib import SequenceMatcher

    if not isinstance(bubbles, list):
        return None
    canonical, _ = check_example(entry, bubbles)
    if canonical is not None:  # already valid: leave the model's answer alone
        return canonical
    sentences = _sentences(entry)
    labels: list[str | None] = [None] * len(entry)
    for bubble in bubbles:
        if not isinstance(bubble, dict):
            continue
        category = bubble.get("category")
        category = category if category in CATEGORIES else "Other"
        texts = bubble.get("texts", bubble.get("text", []))
        for text in [texts] if isinstance(texts, str) else texts if isinstance(texts, list) else []:
            text = text.strip() if isinstance(text, str) else ""
            if not _WORD.search(text):
                continue
            start = entry.find(text)
            # prefer an occurrence whose words are not claimed yet
            while start != -1 and labels[start] is not None and entry.find(text, start + 1) != -1:
                start = entry.find(text, start + 1)
            if start != -1:
                end = start + len(text)
            else:
                scored = [(SequenceMatcher(None, text.lower(), entry[s:e].lower()).ratio(), s, e) for s, e in sentences]
                ratio, start, end = max(scored, default=(0.0, -1, -1))
                if ratio < 0.75:
                    continue
            for i in range(start, end):
                if labels[i] is None:
                    labels[i] = category

    if not any(labels):
        return None
    # fill gaps from the left neighbour, then a leading gap from the right
    last = None
    for i, label in enumerate(labels):
        last = labels[i] = label or last
    first = next(label for label in labels if label)
    labels = [label or first for label in labels]

    # cut into runs of one category, then at sentence ends, trimming whitespace
    pieces: list[tuple[str, str]] = []
    run_start = 0
    for i in range(1, len(entry) + 1):
        if i == len(entry) or labels[i] != labels[run_start]:
            run = entry[run_start:i]
            offset = run_start
            for match in _SENTENCE.finditer(run):
                text = match.group().strip()
                if _WORD.search(text):
                    pieces.append((labels[offset + match.start()], text))  # type: ignore[arg-type]
            run_start = i
    grouped: dict[str, list[str]] = {}
    for category, text in pieces:
        grouped.setdefault(category, []).append(text)
    canonical, _ = check_example(entry, [{"category": c, "texts": t} for c, t in grouped.items()])
    return canonical


def word_categories(entry: str, canonical: list[dict]) -> list[str | None]:
    """Category of each word in the entry, for valid (canonical) bubbles."""
    flat = [(b["category"], t) for b in canonical for t in b["texts"]]
    positions = _locate(entry, [t for _, t in flat]) or []
    labels: list[str | None] = [None] * len(entry)
    for (category, _), (start, end) in zip(flat, positions):
        labels[start:end] = [category] * (end - start)
    return [labels[m.start()] for m in re.finditer(r"\w+", entry)]


def to_chat_record(entry: str, bubbles: list[dict]) -> dict:
    return {"messages": [
        {"role": "user", "content": user_message(entry)},
        {"role": "assistant", "content": json.dumps({"bubbles": bubbles}, ensure_ascii=False)},
    ]}


def from_chat_record(record: dict) -> tuple[str, Any]:
    """Inverse of to_chat_record. Raises on malformed records. Bubbles are unvalidated."""
    user, assistant = record["messages"][0]["content"], record["messages"][1]["content"]
    prefix = f"{INSTRUCTION}\n\n"
    if not user.startswith(prefix):
        raise ValueError("user message does not start with the instruction")
    return user[len(prefix):], json.loads(assistant)["bubbles"]


In [ ]:
%%writefile script.py
"""LoRA fine-tune Gemma 3 270M to sort journal entries into Orbit bubbles.

Local smoke test (CPU, a few steps on the samples, checks the whole pipeline):
    .venv/Scripts/python script.py --smoke

Full run (Colab GPU):
    python script.py --train ../dataset/orbit_bubbles_v2.clean.train.jsonl --val ../dataset/orbit_bubbles_v2.clean.val.jsonl

Outputs in --out: adapter/ (LoRA weights), merged/ (full model for LiteRT export),
eval.json (generation metrics on the validation set).
"""
import argparse
import json
import math
from pathlib import Path

import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

from orbit_schema import check_example, from_chat_record, load_env, repair_bubbles, word_categories

HERE = Path(__file__).parent
IGNORE = -100


def read_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def encode(tokenizer, record: dict, max_len: int) -> dict | None:
    """Tokenize with Gemma's chat template; loss only on the assistant reply."""
    prompt = tokenizer.apply_chat_template(record["messages"][:1], tokenize=False, add_generation_prompt=True)
    full = tokenizer.apply_chat_template(record["messages"], tokenize=False)
    if not full.startswith(prompt):
        raise ValueError("chat template prompt is not a prefix of the full conversation")
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full, add_special_tokens=False)["input_ids"]
    if len(full_ids) > max_len:
        return None
    return {"input_ids": full_ids, "labels": [IGNORE] * len(prompt_ids) + full_ids[len(prompt_ids):]}


class Collator:
    def __init__(self, pad_id: int):
        self.pad_id = pad_id

    def __call__(self, batch: list[dict]) -> dict:
        width = max(len(item["input_ids"]) for item in batch)
        pad = lambda seq, value: seq + [value] * (width - len(seq))
        return {
            "input_ids": torch.tensor([pad(item["input_ids"], self.pad_id) for item in batch]),
            "labels": torch.tensor([pad(item["labels"], IGNORE) for item in batch]),
            "attention_mask": torch.tensor([pad([1] * len(item["input_ids"]), 0) for item in batch]),
        }


@torch.no_grad()
def evaluate(model, tokenizer, records: list[dict], batch_size: int, max_new_tokens: int) -> dict:
    """Greedy-generate on held-out entries and score them the way the app will see them."""
    model.eval()
    tokenizer.padding_side = "left"
    totals = {"n": 0, "valid_json": 0, "passes_rules": 0, "repaired": 0, "exact_match": 0,
              "tp": 0, "pred": 0, "gold": 0, "words": 0, "words_right": 0}
    samples, predictions = [], []
    for start in range(0, len(records), batch_size):
        chunk = records[start:start + batch_size]
        prompts = [tokenizer.apply_chat_template(r["messages"][:1], tokenize=False, add_generation_prompt=True) for r in chunk]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, max_length=None, do_sample=False)
        replies = tokenizer.batch_decode(output[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        for record, reply in zip(chunk, replies):
            entry, gold = from_chat_record(record)
            gold_pairs = {(b["category"], t) for b in gold for t in b["texts"]}
            gold_words = word_categories(entry, gold)
            totals["n"] += 1
            totals["gold"] += len(gold_pairs)
            totals["words"] += len(gold_words)
            predictions.append({"entry": entry, "gold": gold, "reply": reply})
            try:
                predicted = json.loads(reply)["bubbles"]
                totals["valid_json"] += 1
            except (json.JSONDecodeError, KeyError, TypeError):
                samples.append({"entry": entry, "reply": reply, "problem": "invalid json"})
                continue
            canonical, reason = check_example(entry, predicted)
            if canonical is None:
                samples.append({"entry": entry, "reply": reply, "problem": reason})
                canonical = repair_bubbles(entry, predicted)  # what the app would show after repair
                if canonical is None:
                    continue
            else:
                totals["passes_rules"] += 1
            totals["repaired"] += 1
            predictions[-1]["final"] = canonical
            totals["words_right"] += sum(p == g for p, g in zip(word_categories(entry, canonical), gold_words))
            pred_pairs = {(b["category"], t) for b in canonical for t in b["texts"]}
            totals["pred"] += len(pred_pairs)
            totals["tp"] += len(pred_pairs & gold_pairs)
            totals["exact_match"] += canonical == gold
    tokenizer.padding_side = "right"
    n = max(totals["n"], 1)
    precision = totals["tp"] / max(totals["pred"], 1)
    recall = totals["tp"] / max(totals["gold"], 1)
    return {
        "examples": totals["n"],
        "valid_json": totals["valid_json"] / n,
        "passes_rules": totals["passes_rules"] / n,  # raw reply: exact substrings, full coverage, known categories
        "usable_after_repair": totals["repaired"] / n,
        # the rest score the final bubbles (raw reply if valid, else repaired)
        "word_category_accuracy": totals["words_right"] / max(totals["words"], 1),
        "exact_match": totals["exact_match"] / n,
        "span_precision": precision,
        "span_recall": recall,
        "span_f1": 2 * precision * recall / max(precision + recall, 1e-9),
        "failures": samples[:20],
        "predictions": predictions,
    }


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", default="google/gemma-3-270m-it")
    parser.add_argument("--train", type=Path, default=HERE.parent / "dataset" / "orbit_bubbles_v2.clean.train.jsonl")
    parser.add_argument("--val", type=Path, default=HERE.parent / "dataset" / "orbit_bubbles_v2.clean.val.jsonl")
    parser.add_argument("--out", type=Path, default=HERE / "runs" / "orbit-lora")
    parser.add_argument("--epochs", type=float, default=3)
    parser.add_argument("--lr", type=float, default=2e-4)
    # Gemma's 262k vocab makes logits huge (batch x tokens x 262144 floats), so keep
    # micro-batches small and reach an effective batch of 16 through accumulation.
    parser.add_argument("--batch", type=int, default=4)
    parser.add_argument("--grad-accum", type=int, default=4)
    parser.add_argument("--rank", type=int, default=16)
    parser.add_argument("--full", action="store_true", help="train all weights instead of LoRA (use a lower --lr, e.g. 5e-5)")
    parser.add_argument("--max-len", type=int, default=1024, help="app runs with a 1024-token context")
    parser.add_argument("--eval-n", type=int, default=200, help="validation entries to generate on after training")
    parser.add_argument("--max-steps", type=int, default=-1)
    parser.add_argument("--limit", type=int, default=0, help="use only the first N train examples (val gets N/4)")
    parser.add_argument("--smoke", action="store_true", help="short CPU run on a slice of the data to check the pipeline")
    args = parser.parse_args()

    if args.smoke:
        if not args.train.exists() or not args.val.exists():
            print(f"{args.train.name} not found, smoke-testing on orbit_samples.jsonl")
            args.train = args.val = HERE / "orbit_samples.jsonl"
        args.out = HERE / "runs" / "smoke"
        args.limit = args.limit or 32
        args.batch, args.grad_accum, args.max_steps, args.eval_n, args.rank = 1, 4, 10, 4, 8

    load_env()  # optional HF_TOKEN from .env when running locally
    cuda = torch.cuda.is_available()
    # T4 has no bf16 and Gemma 3 overflows in fp16, so fall back to fp32 there.
    bf16 = cuda and torch.cuda.is_bf16_supported()
    dtype = torch.bfloat16 if bf16 else torch.float32
    print(f"device: {'cuda ' + torch.cuda.get_device_name() if cuda else 'cpu'}, dtype: {dtype}")

    tokenizer = AutoTokenizer.from_pretrained(args.model)
    model = AutoModelForCausalLM.from_pretrained(args.model, dtype=dtype, attn_implementation="eager")
    if cuda:
        model.to("cuda")

    records = {"train": read_jsonl(args.train), "val": read_jsonl(args.val)}
    if args.limit:
        records = {"train": records["train"][:args.limit], "val": records["val"][:max(2, args.limit // 4)]}
    datasets = {}
    for name in ("train", "val"):
        encoded = [encode(tokenizer, record, args.max_len) for record in records[name]]
        datasets[name] = [item for item in encoded if item is not None]
        lengths = sorted(len(item["input_ids"]) for item in datasets[name])
        print(f"{name}: {len(datasets[name])} examples ({len(encoded) - len(datasets[name])} over {args.max_len} tokens dropped), "
              f"tokens median {lengths[len(lengths) // 2]} max {lengths[-1]}")

    if args.full:
        print(f"full fine-tune: {sum(p.numel() for p in model.parameters()):,} trainable params")
    else:
        model = get_peft_model(model, LoraConfig(
            r=args.rank, lora_alpha=args.rank * 2, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM",
        ))
        model.print_trainable_parameters()

    steps_per_epoch = math.ceil(len(datasets["train"]) / (args.batch * args.grad_accum))
    total_steps = args.max_steps if args.max_steps > 0 else math.ceil(steps_per_epoch * args.epochs)
    training_args = TrainingArguments(
        output_dir=str(args.out / "checkpoints"),
        num_train_epochs=args.epochs,
        max_steps=args.max_steps,
        per_device_train_batch_size=args.batch,
        per_device_eval_batch_size=args.batch,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        lr_scheduler_type="cosine",
        warmup_steps=max(1, total_steps // 20),
        weight_decay=0.0,
        logging_steps=max(1, min(10, total_steps // 10)),
        eval_strategy="steps" if args.smoke else "epoch",
        eval_steps=total_steps if args.smoke else None,
        save_strategy="no" if args.smoke else "epoch",
        save_total_limit=1,
        load_best_model_at_end=not args.smoke,
        metric_for_best_model="eval_loss",
        bf16=bf16,
        report_to="none",
        remove_unused_columns=False,
        dataloader_pin_memory=cuda,
    )
    trainer = Trainer(
        model=model, args=training_args, data_collator=Collator(tokenizer.pad_token_id),
        train_dataset=datasets["train"], eval_dataset=datasets["val"],
    )
    # Resume from the last saved epoch if a previous run was cut off (e.g. Colab disconnect).
    checkpoints = args.out / "checkpoints"
    last = get_last_checkpoint(str(checkpoints)) if checkpoints.is_dir() and not args.smoke else None
    if last:
        print(f"resuming from {last}")
    trainer.train(resume_from_checkpoint=last)

    if not args.full:
        model.save_pretrained(args.out / "adapter")
    metrics = evaluate(model, tokenizer, records["val"][:args.eval_n], batch_size=args.batch, max_new_tokens=args.max_len // 2)
    (args.out / "eval.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
    print(json.dumps({k: v for k, v in metrics.items() if k not in ("failures", "predictions")}, indent=2))

    merged = model if args.full else model.merge_and_unload()
    merged.save_pretrained(args.out / "merged")
    tokenizer.save_pretrained(args.out / "merged")
    print(f"saved {'' if args.full else 'adapter, '}merged model, and eval.json to {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile export_litert.py
"""Convert the merged fine-tuned model to a .litertlm file for react-native-litert-lm.

Run in Colab (Linux) after script.py:
    pip install litert-torch
    python export_litert.py --merged runs/orbit-lora/merged --out runs/orbit-lora/litert

Settings mirror the app's LLMConfig in src/services/gemma-test.ts (CPU, 1024-token context)
and the q8 build it ships today.
"""
import argparse
import subprocess
import sys
from pathlib import Path


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--merged", type=Path, default=Path("runs/orbit-lora/merged"))
    parser.add_argument("--out", type=Path, default=Path("runs/orbit-lora/litert"))
    parser.add_argument("--cache-length", type=int, default=1024, help="must cover the app's maxContextTokens")
    parser.add_argument("--prefill-lengths", default="256,512", help="entry prompts are well under 512 tokens")
    parser.add_argument("--quantization", default="dynamic_wi8_afp32", help="int8 weights, like the app's q8 model")
    args = parser.parse_args()

    command = [
        sys.executable, "-m", "litert_torch.generative.export_hf",
        f"--model={args.merged}",
        f"--output_dir={args.out}",
        f"--cache_length={args.cache_length}",
        f"--prefill_lengths=[{args.prefill_lengths}]",
        f"--quantization_recipe={args.quantization}",
        "--bundle_litert_lm=True",
    ]
    print(" ".join(command))
    subprocess.run(command, check=True)
    for path in sorted(args.out.rglob("*.litertlm")):
        print(f"{path}  {path.stat().st_size} bytes")


if __name__ == "__main__":
    main()


## 3. Train
Skipped if this run already has a merged model. Resumes from the last checkpoint after a disconnect.

In [ ]:
if os.path.exists(f'{RUN}/merged/model.safetensors') and os.path.exists(f'{RUN}/eval.json'):
    print(f'{RUN_NAME} is already trained; skipping. Use a new RUN_NAME to train again.')
else:
    mode = '--full' if FULL_FINETUNE else f'--rank {RANK}'
    !python script.py {mode} --train {DATA}/{TRAIN_FILE} --val {DATA}/{VAL_FILE} --out {RUN} --epochs {EPOCHS} --batch {BATCH} --grad-accum {GRAD_ACCUM} --lr {LR} --model {BASE_MODEL}

## 4. Evaluation
`usable_after_repair` is what the app will see; `word_category_accuracy` is how often words land in the same bubble as the reference labels.

In [ ]:
def show_metrics(run_name, failures=8):
    metrics = json.load(open(f'/content/drive/MyDrive/orbit/runs/{run_name}/eval.json'))
    print(run_name, json.dumps({k: v for k, v in metrics.items() if k not in ('failures', 'predictions')}, indent=2))
    for failure in metrics['failures'][:failures]:
        print('---', failure['problem'])
        print(failure['entry'])
        print('>>', failure['reply'])
        print()

show_metrics(RUN_NAME)

In [ ]:
# Compare every finished run side by side
runs_dir = '/content/drive/MyDrive/orbit/runs'
keys = ['passes_rules', 'usable_after_repair', 'word_category_accuracy', 'exact_match', 'span_f1']
print(f"{'run':24}" + ''.join(f'{k[:14]:>16}' for k in keys))
for name in sorted(os.listdir(runs_dir)):
    path = f'{runs_dir}/{name}/eval.json'
    if os.path.exists(path):
        m = json.load(open(path))
        print(f'{name:24}' + ''.join(f"{m.get(k, float('nan')):>16.3f}" for k in keys))

## 5. Export to `.litertlm`
Int8 weights and a 1024-token context, matching the app's LiteRT-LM settings. Skipped if the export already exists.

In [ ]:
import glob
if glob.glob(f'{RUN}/litert/**/*.litertlm', recursive=True):
    print('already exported:', glob.glob(f'{RUN}/litert/**/*.litertlm', recursive=True))
else:
    !pip install -q litert-torch
    # litert-lm-builder ships protobuf 7 generated code but does not pin protobuf, and the
    # install downgrades typing-extensions; bring both up to versions everything accepts.
    !pip install -q -U "protobuf>=7.35.1" "typing-extensions>=4.14.1"
    # litert-torch 0.9.4 needs Cache.get_max_cache_shape, which transformers removed in 5.20
    !pip install -q "transformers==5.19.0"
    !python export_litert.py --merged {RUN}/merged --out {RUN}/litert

## 6. Try your own entry

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from orbit_schema import user_message, repair_bubbles

tok = AutoTokenizer.from_pretrained(f'{RUN}/merged')
model = AutoModelForCausalLM.from_pretrained(f'{RUN}/merged', dtype=torch.float32).to('cuda').eval()

def sort_entry(entry):
    prompt = tok.apply_chat_template([{'role': 'user', 'content': user_message(entry)}], tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors='pt', add_special_tokens=False).to('cuda')
    out = model.generate(**ids, max_new_tokens=512, max_length=None, do_sample=False)
    reply = tok.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True)
    try:
        bubbles = repair_bubbles(entry, json.loads(reply)['bubbles'])
    except (json.JSONDecodeError, KeyError, TypeError):
        bubbles = None
    print(entry)
    for bubble in bubbles or []:
        print(f"  {bubble['category']}: {bubble['texts']}")
    if bubbles is None:
        print('  (unusable reply)', reply)
    print()

sort_entry("long day at work and my back hurts. gotta call grandma tmrw its her birthday")
sort_entry("grateful for sunny days")